# Generate and embed IDRs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/generate_and_embed.ipynb)

IDiom generates intrinsically disordered regions two ways — **unprompted** (de novo, no context) and
**prompted** (in-filling an IDR between its flanks) — and exposes the residual stream as embeddings
for downstream models. This notebook covers both.

A GPU is recommended; `DEVICE = "auto"` falls back to CPU.

> **Runtime → Change runtime type → GPU** in Colab before running.

In [ ]:
# Install IDiom if it is not already available (Colab, or a fresh environment).
# Takes a couple of minutes the first time; it pulls lightning, hydra and wandb too.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/rotskoff-group/idiom.git"])

from huggingface_hub import hf_hub_download

import idiom


def example_data(name: str) -> str:
    """Download one cookbook example file from the Hub and return its local path."""
    return hf_hub_download("jxliu2/idiom-data", f"example_data/{name}", repo_type="dataset")


print("idiom", idiom.__file__)

## Parameters

Edit these to point at another model or layer.

In [ ]:
MODEL = "jxliu2/idiom-300M"   # HF repo id, a released directory, or a .ckpt
DEVICE = "auto"               # auto | cpu | cuda
LAYER = 18                    # residual-stream layer to read embeddings from
N = 10                        # sequences per example

In [ ]:
from idiom import IDiom

model = IDiom.from_pretrained(MODEL, device=DEVICE)
print(model)

## Unprompted: de novo IDRs

No context at all — the model samples from what it learned an IDR looks like.

In [ ]:
idrs = model.generate_unprompted(n=N, temperature=1.0, seed=0)

print(f"{len(idrs)} IDRs, mean length {sum(map(len, idrs)) / len(idrs):.0f}")
for s in idrs[:3]:
    print(f"  {len(s):>4}  {s[:70]}{'...' if len(s) > 70 else ''}")

Passing a `length_range` oversamples and then filters, so you get IDRs in the size range you asked
for rather than the size the model felt like.

In [ ]:
sized = model.generate_unprompted(n=N, length_range=(60, 100), seed=0)

print(f"{len(sized)} IDRs, lengths {sorted(len(s) for s in sized)}")

## Prompted: in-fill an IDR between its flanks

Give the model a protein and the coordinates of the disordered span (0-based, half-open) and it
generates replacements for that span, conditioned on the flanking sequence.

In [ ]:
protein = "MEDSKVDNRPQACDEFGHIKLMNPQRSTVWYACDEFGHIKLMNPQRST"
idr_start, idr_end = 12, 30

filled = model.generate_prompted(protein, idr_start, idr_end, n=N, seed=0)

print(f"left flank  ...{protein[:idr_start][-8:]}")
print(f"right flank {protein[idr_end:][:8]}...")
print(f"\n{len(filled)} in-fills:")
for s in filled[:3]:
    print(f"  {len(s):>4}  {s[:70]}")

### Redesigning a real protein

`disprot/` records carry real flanking sequence, which is what prompted generation is for. Passing
`return_full=True` splices each generated IDR back between its flanks and writes the **whole
protein** with a corrected span — how you redesign the IDR of an existing protein, rather than
collecting IDRs on their own.

In [ ]:
from idiom.data.io import read_records

disprot = example_data("disprot/disprot_len1020_idrs.fasta")
record = next(read_records(disprot))

print(f"{record.accession}: {len(record.full_seq)} residues, "
      f"IDR {record.idr_start}-{record.idr_end} ({record.idr_end - record.idr_start} residues)")

redesigned = model.generate_prompted(record.full_seq, record.idr_start, record.idr_end, n=3, seed=0)
for s in redesigned:
    print(f"  {len(s):>4}  {s[:70]}{'...' if len(s) > 70 else ''}")

Both modes have a FASTA form for generating a set to disk — `generate_unprompted_fasta` and
`generate_prompted_fasta`. That output is what the SFT and enrichment workflows consume.

In [ ]:
import tempfile
from pathlib import Path

out = Path(tempfile.mkdtemp()) / "designed.fasta"
model.generate_unprompted_fasta(out, n=N, seed=0)
print(f"wrote {out}\n")
print(out.read_text()[:300])

## Embeddings

`embed` reads the residual stream at whichever layers you ask for. `pool="mean"` gives one vector
per sequence — the usual input to a downstream predictor.

In [ ]:
SEQS = [
    "MEDSKVDNRPQACDEFGHIKLMNPQRSTVWY",
    "GSGSQPQPQPGSGSGSNNNNQQQQGSGSGS",
]

pooled, index = model.embed(SEQS, layers=[LAYER], pool="mean")[LAYER]

print(f"pooled: {pooled.shape} -- one {pooled.shape[1]}-d vector per sequence")
print("accessions:", [r["accession"] for r in index])

`pool="none"` keeps every residue as its own row. The index carries each row's source position, so
rows stay aligned to residues after any filtering.

In [ ]:
per_res, index = model.embed(SEQS[0], layers=[LAYER], pool="none")[LAYER]

print(f"per-residue: {per_res.shape} for a {len(SEQS[0])}-residue sequence")
print("first row:", {k: index[0][k] for k in ("accession", "source_pos", "residue")})

## Next

- **[`sae_features.ipynb`](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/sae_features.ipynb)** — read and steer the
  features behind these embeddings with a sparse autoencoder.
- **`cookbook/scripts/`** — pretraining, SFT, and RL templates for real runs.